In [1]:
from pyspark.sql import SparkSession

# Initialize your Spark session
spark = SparkSession.builder \
    .appName("JovyanSparkTest") \
    .getOrCreate()

# Create a small test DataFrame
data = [("Jovyan", 1), ("Spark", 2)]
df = spark.createDataFrame(data, ["Name", "ID"])
df.show()


+------+---+
|  Name| ID|
+------+---+
|Jovyan|  1|
| Spark|  2|
+------+---+



In [4]:
df = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

df.show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
| John| 22|   80|
| Mary| 25|   90|
|Peter| 30|   70|
+-----+---+-----+



In [6]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Score: integer (nullable = true)



In [7]:
df.select("Name").show()

+-----+
| Name|
+-----+
| John|
| Mary|
|Peter|
+-----+



In [8]:
df.select("Name", "Score").show()

+-----+-----+
| Name|Score|
+-----+-----+
| John|   80|
| Mary|   90|
|Peter|   70|
+-----+-----+



In [9]:
df.filter(df.Score > 80).show()

+----+---+-----+
|Name|Age|Score|
+----+---+-----+
|Mary| 25|   90|
+----+---+-----+



In [10]:
from pyspark.sql.functions import col

df.filter(col("Score") > 80).show()

+----+---+-----+
|Name|Age|Score|
+----+---+-----+
|Mary| 25|   90|
+----+---+-----+



In [34]:
from pyspark.sql.functions import col

# Adding New Columns

df = df.withColumn("Bonus",col("Score") + 5)

df.show()

+-----+---+-----+-----+
| Name|Age|Score|Bonus|
+-----+---+-----+-----+
| John| 22|   80|   85|
| Mary| 25|   90|   95|
|Peter| 30|   70|   75|
+-----+---+-----+-----+



In [13]:
# ascending
df.orderBy("Score").show()

+-----+---+-----+-----+
| Name|Age|Score|Bonus|
+-----+---+-----+-----+
|Peter| 30|   70|   75|
| John| 22|   80|   85|
| Mary| 25|   90|   95|
+-----+---+-----+-----+



In [14]:
# descending

df.orderBy(df.Score.desc()).show()

+-----+---+-----+-----+
| Name|Age|Score|Bonus|
+-----+---+-----+-----+
| Mary| 25|   90|   95|
| John| 22|   80|   85|
|Peter| 30|   70|   75|
+-----+---+-----+-----+



In [15]:
# aggregation

from pyspark.sql.functions import avg

df.select(avg("Score")).show()

+----------+
|avg(Score)|
+----------+
|      80.0|
+----------+



In [16]:
# maximum score

from pyspark.sql.functions import max

df.select(max("Score")).show()

+----------+
|max(Score)|
+----------+
|        90|
+----------+



## GroupBy

In [19]:
data = [
    ("CS", 80),
    ("CS", 90),
    ("Math", 75),
    ("Math", 85)
]

df = spark.createDataFrame(data,["Department", "Score"])

df.groupBy("Department").avg("Score").show()

+----------+----------+
|Department|avg(Score)|
+----------+----------+
|        CS|      85.0|
|      Math|      80.0|
+----------+----------+



## Missing values

In [21]:
df.na.drop().show()

+----------+-----+
|Department|Score|
+----------+-----+
|        CS|   80|
|        CS|   90|
|      Math|   75|
|      Math|   85|
+----------+-----+



In [22]:
df.na.fill(0).show()

+----------+-----+
|Department|Score|
+----------+-----+
|        CS|   80|
|        CS|   90|
|      Math|   75|
|      Math|   85|
+----------+-----+



## Spark SQL

In [25]:
df = spark.read.csv(
    "students.csv",
    header=True,
    inferSchema=True
)

df.createOrReplaceTempView("students")

In [26]:
spark.sql("""
SELECT Name, Score
FROM students
WHERE Score > 80
""").show()

+----+-----+
|Name|Score|
+----+-----+
|Mary|   90|
+----+-----+



## Joins

In [29]:
students = [
    (1, "John"),
    (2, "Mary")
]

scores = [
    (1, 80),
    (2, 90)
]

# create dataframe

students_df = spark.createDataFrame(
    students,
    ["id", "name"]
)

scores_df = spark.createDataFrame(
    scores,
    ["id", "score"]
)

df_join = students_df.join(scores_df,"id")

df_join.show()

+---+----+-----+
| id|name|score|
+---+----+-----+
|  1|John|   80|
|  2|Mary|   90|
+---+----+-----+



## Spark Transformations

- select()
- filter()
- groupBy()
- join()
- withColumn()

## Spark Actions
- show()
- count()
- collect()
- take()
- write()

In [30]:
df.collect()

[Row(Name='John', Age=22, Score=80),
 Row(Name='Mary', Age=25, Score=90),
 Row(Name='Peter', Age=30, Score=70)]

In [31]:
df.take(1)

[Row(Name='John', Age=22, Score=80)]

In [33]:
df.repartition(4)

DataFrame[Name: string, Age: int, Score: int]

## Write to file

In [32]:
df_join.write.csv(
    "output/",
    header=True
)

## Read dataset

In [36]:
df = spark.read.csv("sales.csv", header=True, inferSchema=True)
df.show()

+-------+----------+-----------+--------+-----+
|OrderID|   Product|   Category|Quantity|Price|
+-------+----------+-----------+--------+-----+
|      1|    Laptop|Electronics|       1| 1200|
|      2|     Phone|Electronics|       2|  800|
|      3|     Chair|  Furniture|       4|  150|
|      4|     Table|  Furniture|       1|  300|
|      5|Headphones|Electronics|       3|  200|
+-------+----------+-----------+--------+-----+



In [37]:
from pyspark.sql.functions import col

df = df.withColumn("Revenue", col("Quantity") * col("Price"))
df.show()

+-------+----------+-----------+--------+-----+-------+
|OrderID|   Product|   Category|Quantity|Price|Revenue|
+-------+----------+-----------+--------+-----+-------+
|      1|    Laptop|Electronics|       1| 1200|   1200|
|      2|     Phone|Electronics|       2|  800|   1600|
|      3|     Chair|  Furniture|       4|  150|    600|
|      4|     Table|  Furniture|       1|  300|    300|
|      5|Headphones|Electronics|       3|  200|    600|
+-------+----------+-----------+--------+-----+-------+



In [38]:
# total revenue by category

df.groupBy("Category") \
  .sum("Revenue") \
  .show()

+-----------+------------+
|   Category|sum(Revenue)|
+-----------+------------+
|Electronics|        3400|
|  Furniture|         900|
+-----------+------------+



In [39]:
# top selling product

df.groupBy("Product") \
  .sum("Quantity") \
  .orderBy("sum(Quantity)", ascending=False) \
  .show()

+----------+-------------+
|   Product|sum(Quantity)|
+----------+-------------+
|     Chair|            4|
|Headphones|            3|
|     Phone|            2|
|    Laptop|            1|
|     Table|            1|
+----------+-------------+



In [40]:
df.limit(2).show()

+-------+-------+-----------+--------+-----+-------+
|OrderID|Product|   Category|Quantity|Price|Revenue|
+-------+-------+-----------+--------+-----+-------+
|      1| Laptop|Electronics|       1| 1200|   1200|
|      2|  Phone|Electronics|       2|  800|   1600|
+-------+-------+-----------+--------+-----+-------+



In [41]:
df.count()

5

In [42]:
df.write \
  .mode("overwrite") \
  .partitionBy("Category") \
  .parquet("output/sales_data")

In [43]:
df.groupBy(df.columns) \
  .count() \
  .filter("count > 1") \
  .show()

+-------+-------+--------+--------+-----+-------+-----+
|OrderID|Product|Category|Quantity|Price|Revenue|count|
+-------+-------+--------+--------+-----+-------+-----+
+-------+-------+--------+--------+-----+-------+-----+



## Tasks:

- Read CSV into Spark.
- Display schema.
- Calculate average score.
- Find top 5 students.
- Calculate average score per department.
- Find students with attendance below 75%.
- Create SQL view and query.
- Export results to CSV.